# 1) Import Configuration and Functions:

In [0]:
%run ../common/configuration


In [0]:
%run ../common/functions

# 2) Define Status Schema:

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

status_schema = StructType([
    StructField("status_id", IntegerType(), False),
    StructField("status", StringType(), True),
    StructField("count", IntegerType(), True),
    
])

status_input_path = f"{processed_folder_path}/status/csv/status.csv"

status_df = spark.read \
    .option("header", True) \
    .schema(status_schema) \
    .csv(status_input_path)


# 3) Transform Status Data:

The steps included:

- Create Surrogate Key.
- Add Data Source and File Date.

In [0]:
from pyspark.sql.functions import lit

status_with_audit_df = status_df \
    .withColumn("data_source", lit(v_data_source)) \
    .withColumn("file_date", lit(v_file_date))

status_date_df = add_ingestion_date(status_with_audit_df)


status_final_df = add_surrogate_key(
    status_date_df,
    key_column_name="status_sk",
    hash_columns=["status_id", "status", "count"],
)

print("Final columns going into the write:", status_final_df.columns)

# 4) Save the Processed Dataset to Delta Lake:

In [0]:
status_output_path = f"{processed_folder_path}/status/delta"

spark.sql("CREATE DATABASE IF NOT EXISTS f1_processed")

upsert_if_changed(
    input_df=status_final_df,
    db_name="f1_processed",
    table_name="status",
    output_path=status_output_path,
    merge_key_columns=["status_id"],
)

In [0]:
display(spark.read.format("delta").load(status_output_path))

In [0]:
build_presentation_fact(
    processed_location=f"{processed_folder_path}/status/delta",
    presentation_directory=f"{presentation_folder_path}/fact_status/delta",
    db_name="f1_presentation",
    table_name="fact_status",
)

In [0]:
display(spark.read.format("delta").load(f"{presentation_folder_path}/fact_status/delta"))

# 5) Save backup Status in CSV format:

In [0]:
import io
import csv

status_backup_path = f"{presentation_folder_path}/fact_status/csv/fact_status.csv"

backup_rows = [row.asDict() for row in status_final_df.collect()]
backup_fieldnames = status_final_df.columns

backup_buffer = io.StringIO()
backup_writer = csv.DictWriter(backup_buffer, fieldnames=backup_fieldnames)
backup_writer.writeheader()
backup_writer.writerows(backup_rows)

dbutils.fs.put(status_backup_path, backup_buffer.getvalue(), overwrite=True)
print(f"backup saved: {status_backup_path}")